# MACCS fingerprints with Data 4.3

## 1. Data ingestion and cleaning

In [ ]:
import pandas as pd

df_cdd = pd.read_csv('data_4.csv')
df_cdd = df_cdd.rename(columns={
    "GU CRC L6: Average EC50": "gu_ec50",
    "GU CRC L6: Average EC50 Standard Deviation (±)": "gu_std",
    "GU CRC L6: Average EC50 Count": "gu_count",
    "GU CRC L6: Average Emax": "gu_emax",
    "B2 cAMP CRC L6: Average Emax Standard Deviation (±)": "gu_emax_std",
    "B2 cAMP CRC L6: Average Emax Count": "gu_emax_count",
    "B2 cAMP CRC L6: Average EC50 2": "camp_ec50",
    'B2 cAMP CRC L6: Average EC50 2 Standard Deviation (±)': "camp_std",
    'B2 cAMP CRC L6: Average EC50 2 Count': "camp_count",
    "B2 cAMP CRC L6: Average Emax": "camp_emax",
    "GU CRC L6: Average Emax Standard Deviation (±)": "camp_emax_std",
    "GU CRC L6: Average Emax Count": "camp_emax_count"
})

In [ ]:
camp_cols = [
    "B2 cAMP CRC L6: cAMP (-5) (of iso)",
    "B2 cAMP CRC L6: cAMP (-6) (of iso)",
    "B2 cAMP CRC L6: cAMP (-7) (of iso)",
    "B2 cAMP CRC L6: cAMP (-8) (of iso)",
    "B2 cAMP CRC L6: cAMP (-9) (of iso)",
    "B2 cAMP CRC L6: cAMP (-10) (of iso)"
]

gu_cols = [
    "GU CRC L6: GU (-5) (of iso)",
    "GU CRC L6: GU (-6) (of iso)",
    "GU CRC L6: GU (-7) (of iso)",
    "GU CRC L6: GU (-8) (of iso)",
    "GU CRC L6: GU (-9) (of iso)",
    "GU CRC L6: GU (-10) (of iso)"
]

df_cdd["camp_emax"] = df_cdd[camp_cols].max(axis=1, skipna=True)
df_cdd["gu_emax"] = df_cdd[gu_cols].max(axis=1, skipna=True)

In [ ]:
df_cdd = df_cdd.drop(['Synonyms',
                      'B2 cAMP CRC L6: Basal (of iso)',
                      'B2 cAMP CRC L6: cAMP Clen (of iso)',
                      "B2 cAMP CRC L6: cAMP (-5) (of iso)",
                      'B2 cAMP CRC L6: cAMP (-6) (of iso)',
                      'B2 cAMP CRC L6: cAMP (-7) (of iso)',
                      'B2 cAMP CRC L6: cAMP (-8) (of iso)',
                      'B2 cAMP CRC L6: cAMP (-9) (of iso)',
                      'B2 cAMP CRC L6: cAMP (-10) (of iso)',
                      'B2 cAMP CRC L6: Top (of iso)',
                      'B2 cAMP CRC L6: Bottom (of iso)',
                      'B2 cAMP CRC L6: Average EC50 batch',
                      'B2 cAMP CRC L6: Average EC50 batch Standard Deviation (±)',
                      'B2 cAMP CRC L6: Average EC50 batch Count',
                      'B2 cAMP CRC L6: Emax',
                      'GU CRC L6: GU (-5) (of iso)',
                      'GU CRC L6: GU (-6) (of iso)',
                      'GU CRC L6: GU (-7) (of iso)',
                      'GU CRC L6: GU (-8) (of iso)',
                      'GU CRC L6: GU (-9) (of iso)',
                      'GU CRC L6: GU (-10) (of iso)',
                      'GU CRC L6: Bottom L6G (of iso)',
                      'GU CRC L6: Top L6G (of iso)',
                      'GU CRC L6: Average EC50 batch',
                      'GU CRC L6: Average EC50 batch Standard Deviation (±)',
                      'GU CRC L6: Average EC50 batch Count',
                      'GU CRC L6: Emax'
                    ], axis=1)


In [ ]:
import numpy as np

# Group by Molecule Name and SMILES, then aggregate
df_cdd = df_cdd.groupby(['Molecule Name', 'SMILES']).agg({
    # For EC50 values: take mean
    'camp_ec50': 'mean',
    'gu_ec50': 'mean',
    
    # For Emax values: take mean
    'camp_emax': 'mean',
    'gu_emax': 'mean',
    
    # For standard deviations: pool them using root mean square
    'camp_std': lambda x: np.sqrt(np.mean(x**2)),
    'gu_std': lambda x: np.sqrt(np.mean(x**2)),
    'gu_emax_std': lambda x: np.sqrt(np.mean(x**2)),
    'camp_emax_std': lambda x: np.sqrt(np.mean(x**2)),
}).reset_index()

In [ ]:
df_cdd = df_cdd.drop(['camp_std',
                      'gu_emax_std',
                      'gu_std',
                      'camp_emax_std'
                    ], axis=1)

df_cdd = df_cdd[["Molecule Name", "SMILES", "gu_ec50", "gu_emax", "camp_ec50", "camp_emax"]]

df_cdd = df_cdd.dropna(subset=[ 'gu_ec50',
                                'camp_ec50' ])

In [ ]:
df = df_cdd
df.columns = df.columns.str.strip()

# Basic sanity check
TARGET_NAMES = ['gu_ec50', 'gu_emax', 'camp_ec50', 'camp_emax']
required = {'Molecule Name', 'SMILES'} | set(TARGET_NAMES)
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'

print(f'Dataset shape: {df.shape}')
print(df[TARGET_NAMES].describe())
df.head()

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from copy import deepcopy

# RDKit
from rdkit import Chem
from rdkit.Chem import MACCSkeys, Draw

# Sklearn
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

# XGBoost
import xgboost as xgb

# SHAP
import shap

SEED = 42
np.random.seed(SEED)
print('All imports successful.')

## 2. MACCS Keys Fingerprints (167-bit)

> MACCS keys 1–166 are used (bit 0 is always 0 in RDKit).

In [ ]:
def smiles_to_maccs(smi):
    mol = Chem.MolFromSmiles(smi)
    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array(fp)

X_maccs = np.vstack(df['SMILES'].apply(smiles_to_maccs).values)  # (N, 167)
y = df[TARGET_NAMES].values  # (N, 4)

# MACCS bit names for later labelling
maccs_names = [f'MACCS_{i}' for i in range(167)]

# Remove always-zero or always-one bits (no variance)
var_mask = (X_maccs.var(axis=0) > 0)
X_filtered = X_maccs[:, var_mask]
names_filtered = [maccs_names[i] for i, v in enumerate(var_mask) if v]

print(f'MACCS shape (raw): {X_maccs.shape}')
print(f'MACCS shape (after removing zero-variance bits): {X_filtered.shape}')

# Visualise bit frequency
bit_freq = X_filtered.mean(axis=0)
plt.figure(figsize=(14,3))
plt.bar(range(len(bit_freq)), bit_freq, color='steelblue')
plt.xlabel('MACCS bit index (filtered)')
plt.ylabel('Frequency')
plt.title('MACCS Key Bit Frequencies across the Dataset')
plt.tight_layout()
plt.show()

In [ ]:
# Extract target variables
target_cols = ['gu_ec50', 'gu_emax', 'camp_ec50', 'camp_emax']
y = df[target_cols].values

print("Target variables shape:", y.shape)

# Visualize target distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 6))
axes = axes.ravel()

for idx, col in enumerate(target_cols):
    axes[idx].hist(df[col], bins=20, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('target_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Cross-Validation Setup

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

def cv_metrics(y_true, y_pred, target_names=TARGET_NAMES):
    """Return per-target RMSE, MAE, R2."""
    results = {}
    for i, name in enumerate(target_names):
        results[name] = {
            'RMSE': np.sqrt(mean_squared_error(y_true[:,i], y_pred[:,i])),
            'MAE':  mean_absolute_error(y_true[:,i], y_pred[:,i]),
            'R2':   r2_score(y_true[:,i], y_pred[:,i])
        }
    return results

def print_cv_summary(all_fold_metrics, model_name):
    print(f'\n{"─"*65}')
    print(f'  {model_name} — 5-Fold CV Summary')
    print(f'{"─"*65}')
    for target in TARGET_NAMES:
        rmse_vals = [m[target]['RMSE'] for m in all_fold_metrics]
        mae_vals  = [m[target]['MAE']  for m in all_fold_metrics]
        r2_vals   = [m[target]['R2']   for m in all_fold_metrics]
        print(f'  [{target.upper()}]  RMSE: {np.mean(rmse_vals):.4f}±{np.std(rmse_vals):.4f} | '
              f'MAE: {np.mean(mae_vals):.4f}±{np.std(mae_vals):.4f} | '
              f'R2: {np.mean(r2_vals):.4f}±{np.std(r2_vals):.4f}')
    print(f'{"─"*65}')

## 4. Machine Learning models 
### 4.1 Multi-output support vector regressor

In [ ]:
svr_fold_metrics = []
svr_fold_models  = []   # store (scaler, model) per fold

for fold, (train_idx, val_idx) in enumerate(kf.split(X_filtered)):
    X_tr, X_val = X_filtered[train_idx], X_filtered[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    scaler = StandardScaler()
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_val_sc = scaler.transform(X_val)

    # MultiOutputRegressor wraps SVR for 4-target regression
    base_svr = SVR(kernel='rbf', C=10, epsilon=0.1, gamma='scale')
    mo_svr   = MultiOutputRegressor(base_svr, n_jobs=-1)
    mo_svr.fit(X_tr_sc, y_tr)

    y_pred = mo_svr.predict(X_val_sc)
    metrics = cv_metrics(y_val, y_pred)
    svr_fold_metrics.append(metrics)
    svr_fold_models.append((scaler, mo_svr, train_idx, val_idx))
    rmse_str = '  '.join([f'{t}_RMSE={metrics[t]["RMSE"]:.4f}' for t in TARGET_NAMES])
    print(f'Fold {fold+1}: {rmse_str}')

print_cv_summary(svr_fold_metrics, 'Multi-Output SVR')

### 4.2 Multi-output random forest

In [ ]:
rf_fold_metrics = []
rf_fold_models  = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_filtered)):
    X_tr, X_val = X_filtered[train_idx], X_filtered[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    rf = RandomForestRegressor(
        n_estimators=300, max_features='sqrt',
        random_state=SEED, n_jobs=-1
    )
    # RandomForestRegressor natively handles multi-output
    rf.fit(X_tr, y_tr)

    y_pred = rf.predict(X_val)
    metrics = cv_metrics(y_val, y_pred)
    rf_fold_metrics.append(metrics)
    rf_fold_models.append((rf, train_idx, val_idx))
    rmse_str = '  '.join([f'{t}_RMSE={metrics[t]["RMSE"]:.4f}' for t in TARGET_NAMES])
    print(f'Fold {fold+1}: {rmse_str}')

print_cv_summary(rf_fold_metrics, 'Multi-Output Random Forest')

### 4.3 Multi-output XGBoost

In [ ]:
xgb_fold_metrics = []
xgb_fold_models  = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_filtered)):
    X_tr, X_val = X_filtered[train_idx], X_filtered[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    # XGBoost >= 2.0 supports multi-output natively with multi_strategy
    xgb_model = xgb.XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        multi_strategy='multi_output_tree',   # native multi-output
        tree_method='hist',
        random_state=SEED,
        n_jobs=-1
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    y_pred = xgb_model.predict(X_val)
    metrics = cv_metrics(y_val, y_pred)
    xgb_fold_metrics.append(metrics)
    xgb_fold_models.append((xgb_model, train_idx, val_idx))
    rmse_str = '  '.join([f'{t}_RMSE={metrics[t]["RMSE"]:.4f}' for t in TARGET_NAMES])
    print(f'Fold {fold+1}: {rmse_str}')

print_cv_summary(xgb_fold_metrics, 'Multi-Output XGBoost')

### 4.4 Model performance comparison

In [ ]:
def aggregate_cv(fold_metrics, model_name):
    rows = []
    for target in ['gu_ec50','gu_emax', 'camp_ec50', 'camp_emax']:
        for metric in ['RMSE','R2']:
            vals = [m[target][metric] for m in fold_metrics]
            rows.append({'Model': model_name, 'Target': target,
                         'Metric': metric, 'Mean': np.mean(vals), 'Std': np.std(vals)})
    return pd.DataFrame(rows)

results_df = pd.concat([
    aggregate_cv(svr_fold_metrics,  'SVR'),
    aggregate_cv(rf_fold_metrics,   'Random Forest'),
    aggregate_cv(xgb_fold_metrics,  'XGBoost'),
], ignore_index=True)

# Pivot for readability
pivot = results_df.pivot_table(index=['Model','Target'], columns='Metric', values=['Mean','Std'])
print(pivot.round(4))

# Bar chart with best model highlighted in green
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, target in zip(axes, ['gu_ec50','gu_emax', 'camp_ec50', 'camp_emax']):
    sub = results_df[(results_df['Target'] == target) & (results_df['Metric'] == 'RMSE')]

    best_idx = sub['Mean'].idxmin()
    best_model = sub.loc[best_idx, 'Model']

    colors = ['green' if m == best_model else 'steelblue' for m in sub['Model']]

    ax.bar(sub['Model'], sub['Mean'], yerr=sub['Std'], capsize=5,
           color=colors, alpha=0.85)

    ax.set_title(f'{target.upper()} — Mean RMSE (5-Fold CV)')
    ax.set_ylabel('RMSE')
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


print('\nExpected output: bar chart + pivot table with RMSE/MAE/R2 for all 4 models on both targets.')

## 5. Helper — Bias Score for High GU + Low cAMP

For each feature $f$, we define a **4-target bias score**:

$$\\text{BiasScore}(f) = -\\phi_{gu\_ec50}(f) + \\phi_{gu\_emax}(f) + \\phi_{camp\_ec50}(f) - \\phi_{camp\_emax}(f)$$

where $\\phi$ are SHAP values. The sign convention reflects that lower EC50 and higher Emax are desirable. A positive bias score means the feature promotes the desired potency/efficacy profile.

> You can adjust `BIAS_SIGNS` below to change which direction is 'good' for each target.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Desired direction per target
BIAS_SIGNS = {
    'gu_ec50':   -1,  # lower better
    'gu_emax':   +1,  # higher better
    'camp_ec50': +1,  # higher better
    'camp_emax': -1,  # lower better
}

# Optional biological weighting
TARGET_WEIGHTS = {
    'gu_ec50':   0.5,
    'gu_emax':   0.5,
    'camp_ec50': 0.5,
    'camp_emax': 0.5,
}


def normalize(v):
    """Min-max normalize vector to [-1,1] preserving sign"""
    vmin, vmax = v.min(), v.max()
    if vmax == vmin:
        return np.zeros_like(v)
    return 2 * (v - vmin) / (vmax - vmin) - 1


def compute_bias(shap_dict, normalize_targets=True):
    """
    Computes pathway bias score.

    Steps:
    1. Mean SHAP per feature per target
    2. Normalize per target (optional)
    3. Apply optimization direction
    4. Aggregate glucose vs cAMP contributions

    Returns
    -------
    bias : (F,) vector
    """

    mean_shap = {t: sv.mean(axis=0) for t, sv in shap_dict.items()}

    if normalize_targets:
        mean_shap = {t: normalize(v) for t, v in mean_shap.items()}

    # Apply desired direction
    adjusted = {
        t: BIAS_SIGNS[t] * TARGET_WEIGHTS[t] * mean_shap[t]
        for t in mean_shap
    }

    # Glucose pathway
    gu_contrib = (
        adjusted['gu_ec50'] +
        adjusted['gu_emax']
    )

    # cAMP pathway
    camp_contrib = (
        adjusted['camp_ec50'] +
        adjusted['camp_emax']
    )

    # Final bias
    bias = gu_contrib - camp_contrib

    return bias


def bias_score_plot(shap_dict, feature_names, top_n=20, title='Glucose vs cAMP Bias Score'):

    bias = compute_bias(shap_dict)

    idx_sorted = np.argsort(np.abs(bias))[::-1][:top_n]

    top_names = [feature_names[i] for i in idx_sorted]
    top_bias = bias[idx_sorted]

    colors = ['#2196F3' if b > 0 else '#F44336' for b in top_bias]

    plt.figure(figsize=(10, 6))
    plt.barh(top_names[::-1], top_bias[::-1], color=colors[::-1])
    plt.axvline(0, color='black', linewidth=0.8)
    plt.xlabel('Bias Score (↑ favors glucose over cAMP)')
    plt.title(title)
    plt.tight_layout()
    plt.show()

    rows = {
        'Feature': top_names,
        'BiasScore': top_bias
    }

    for t in shap_dict.keys():
        rows[f'MeanSHAP_{t}'] = shap_dict[t].mean(axis=0)[idx_sorted]

    return pd.DataFrame(rows)


print("Improved pharmacological bias-score helper defined.")

## 6. Explainable AI methods
### 6.1 KernelSHAP — SVR

KernelSHAP is model-agnostic and the appropriate SHAP method for SVR.

In [ ]:
# Use best fold (lowest mean RMSE across all targets)
best_fold_svr = np.argmin([
    np.mean([m[t]['RMSE'] for t in TARGET_NAMES]) for m in svr_fold_metrics
])
scaler_svr, mo_svr_best, tr_idx, val_idx = svr_fold_models[best_fold_svr]

X_tr_sc_svr  = scaler_svr.transform(X_filtered[tr_idx])
X_val_sc_svr = scaler_svr.transform(X_filtered[val_idx])

# KernelSHAP uses a background summary (k-means to keep it tractable)
background = shap.kmeans(X_tr_sc_svr, 30)

shap_svr = {}  # {target: (N, F) array}

for i, (estimator, target) in enumerate(zip(mo_svr_best.estimators_, TARGET_NAMES)):
    print(f'  Computing KernelSHAP for SVR output {i+1}/{len(TARGET_NAMES)}: {target} ...')
    explainer = shap.KernelExplainer(estimator.predict, background)
    sv = explainer.shap_values(X_val_sc_svr, nsamples=200)
    shap_svr[target] = np.array(sv)

# ── Individual SHAP summary plots ─────────────────────────────────────────
for target in TARGET_NAMES:
    print(f'\n[SVR] SHAP Summary — {target}:')
    shap.summary_plot(shap_svr[target], X_val_sc_svr, feature_names=names_filtered,
                      max_display=20, plot_type='bar', show=True)

# ── Bias score ─────────────────────────────────────────────────────────────
print('\n[SVR] Bias Score — Desired Activity Profile:')
svr_bias_df = bias_score_plot(
    shap_svr, names_filtered, top_n=20,
    title='SVR — Bias Score (Desired Activity Profile)'
)
display(svr_bias_df.head(10))

### 6.2 TreeSHAP — Random Forest

In [ ]:
best_fold_rf = np.argmin([
    np.mean([m[t]['RMSE'] for t in TARGET_NAMES]) for m in rf_fold_metrics
])
rf_best, tr_idx_rf, val_idx_rf = rf_fold_models[best_fold_rf]

X_tr_rf  = X_filtered[tr_idx_rf]
X_val_rf = X_filtered[val_idx_rf]

# ── (a) MDI Feature Importance ─────────────────────────────────────────────
n_feats = X_tr_rf.shape[1]
fi_per_output = np.zeros((len(TARGET_NAMES), n_feats))
for tree in rf_best.estimators_:
    fi_per_output += tree.feature_importances_.reshape(1, n_feats)
fi_per_output /= len(rf_best.estimators_)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (i, target) in zip(axes.flat, enumerate(TARGET_NAMES)):
    top20_idx = np.argsort(fi_per_output[i])[::-1][:20]
    ax.barh([names_filtered[j] for j in top20_idx][::-1],
             fi_per_output[i][top20_idx][::-1], color='teal')
    ax.set_title(f'RF MDI Importance — {target}')
    ax.set_xlabel('Importance')
plt.suptitle('Random Forest Feature Importance (MDI)', fontweight='bold')
plt.tight_layout()
plt.show()

# ── (b) TreeSHAP ───────────────────────────────────────────────────────────
rf_explainer = shap.TreeExplainer(rf_best)
shap_rf_vals = rf_explainer.shap_values(X_val_rf)  # list of n_outputs arrays (N, F)

shap_rf = {}
if isinstance(shap_rf_vals, list):
    for i, target in enumerate(TARGET_NAMES):
        shap_rf[target] = np.array(shap_rf_vals[i])
elif shap_rf_vals.ndim == 3:
    for i, target in enumerate(TARGET_NAMES):
        shap_rf[target] = shap_rf_vals[:, :, i]
else:
    raise ValueError('Unexpected SHAP output shape from RF TreeExplainer')


print('\n[RF] Bias Score — Desired Activity Profile:')
rf_bias_df = bias_score_plot(
    shap_rf, names_filtered, top_n=20,
    title='Random Forest — Bias Score (Desired Activity Profile)'
)
display(rf_bias_df.head(10))

### 6.3 TreeSHAP + Feature Importance — XGBoost

In [ ]:
best_fold_xgb = np.argmin([
    np.mean([m[t]['RMSE'] for t in TARGET_NAMES]) for m in xgb_fold_metrics
])
xgb_best, tr_idx_xgb, val_idx_xgb = xgb_fold_models[best_fold_xgb]

X_tr_xgb  = X_filtered[tr_idx_xgb]
X_val_xgb = X_filtered[val_idx_xgb]

# ── (a) Built-in feature importance ───────────────────────────────────────
# NOTE: 'gain' importance is not supported for multi_output_tree in XGBoost.
# We use 'weight' (split frequency) instead, which is supported.
fi_xgb = xgb_best.get_booster().get_score(importance_type='weight')

# get_score() returns a dict {feature_name: score} and may omit zero-importance
# features — remap to a full-length array aligned with maccs_names_filtered
n_feats = X_tr_xgb.shape[1]
fi_xgb_arr = np.zeros(n_feats)
for fname, score in fi_xgb.items():
    # XGBoost uses 'f0', 'f1', ... when no feature names are set
    idx = int(fname.replace('f', ''))
    fi_xgb_arr[idx] = score

top20_xgb = np.argsort(fi_xgb_arr)[::-1][:20]
plt.figure(figsize=(10, 6))
plt.barh([names_filtered[i] for i in top20_xgb][::-1],
          fi_xgb_arr[top20_xgb][::-1], color='darkorange')
plt.xlabel('Weight Importance (split count)')
plt.title('XGBoost Feature Importance (Multi-Output, weight)')
plt.tight_layout()
plt.show()

# ── (b) KernelSHAP per output ──────────────────────────────────────────────
# NOTE: shap.TreeExplainer does not support XGBoost's multi_output_tree
# (vector-leaf trees) — raises "vector-leaf is not yet supported".
# KernelSHAP is model-agnostic and works correctly here.
# We use a k-means background summary to keep it tractable.

print('Computing KernelSHAP for XGBoost (this may take a few minutes)...')
bg_xgb = shap.kmeans(X_tr_xgb, 30)

shap_xgb = {}
for i, target in enumerate(TARGET_NAMES):
    print(f'  Output {i+1}/{len(TARGET_NAMES)}: {target} ...')
    def predict_output(x, idx=i):
        return xgb_best.predict(x)[:, idx]
    ex = shap.KernelExplainer(predict_output, bg_xgb)
    shap_xgb[target] = ex.shap_values(X_val_xgb, nsamples=200)

print('\n[XGB] Bias Score — Desired Activity Profile:')
xgb_bias_df = bias_score_plot(
    shap_xgb, names_filtered, top_n=20,
    title='XGBoost — Bias Score (Desired Activity Profile)'
)
display(xgb_bias_df.head(10))

## 7. Consensus Bias Score Across Models

Aggregate MACCS-level bias scores from SVR, RF, and XGBoost to find features that **all models agree** drive high GU + low cAMP.

In [ ]:
def get_bias_vector(shap_dict):
    """Returns (F,) bias vector from shap_dict."""
    return compute_bias(shap_dict)

bias_svr  = get_bias_vector(shap_svr)
bias_rf   = get_bias_vector(shap_rf)
bias_xgb  = get_bias_vector(shap_xgb)

# Normalise each to [-1, 1] before averaging
def normalise(v):
    m = np.abs(v).max()
    return v / m if m > 0 else v

consensus_bias = (normalise(bias_svr) + normalise(bias_rf) + normalise(bias_xgb)) / 3.0

top_n = 25
idx_top = np.argsort(np.abs(consensus_bias))[::-1][:top_n]
names_top = [names_filtered[i] for i in idx_top]
vals_top  = consensus_bias[idx_top]

colors = ['#1976D2' if v > 0 else '#D32F2F' for v in vals_top]
plt.figure(figsize=(11, 7))
plt.barh(names_top[::-1], vals_top[::-1], color=colors[::-1])
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Normalised Consensus Bias Score')
plt.title('Consensus Bias Score (SVR + RF + XGBoost)\n↑ = Desired Activity Profile (Low EC50 & High Emax)', fontsize=13)
plt.tight_layout()
plt.show()

consensus_df = pd.DataFrame({
    'Feature':        names_top,
    'ConsensusScore': vals_top,
    'SVR_bias':  bias_svr[idx_top],
    'RF_bias':   bias_rf[idx_top],
    'XGB_bias':  bias_xgb[idx_top]
}).sort_values('ConsensusScore', ascending=False)

print('Top features driving desired activity profile (positive score):')
display(consensus_df[consensus_df['ConsensusScore'] > 0].head(10))
print('\nTop features that hinder the desired outcome (negative score):')
display(consensus_df[consensus_df['ConsensusScore'] < 0].head(10))

## 8. MACCS Key Interpretation Helper

Map top-scoring MACCS bit indices back to their chemical meaning.

In [ ]:
# ============================================================
# 1. FULL MACCS KEY DESCRIPTION DICTIONARY
#    (Your original entries + all missing bits you requested)
# ============================================================

MACCS_DESCRIPTIONS = {
    1: 'ISOTOPE',
    2: 'Atomic num > 103 (rare element)',
    3: 'Group IVA, VA, VIA periods 4+ (Ge, Sn, Pb, As, Sb, Bi, Se, Te)',
    4: 'Actinide',
    5: 'Atom with atomic num > 103',
    6: 'Group IIIB,IVB (Sc, Ti, La, Hf, ...)',
    7: 'Lanthanide',
    8: 'Group VB, VIB, VIIB (V, Cr, Mn, Nb, Mo, Tc, Ta, W, Re)',
    9: 'QAAA@1 (4-atom chain)',
    10: 'Group VIII (Fe, Co, Ni, Ru, Rh, Pd, Os, Ir, Pt)',
    11: 'Group IIA (Alkaline earth: Be, Mg, Ca, Sr, Ba, Ra)',
    12: '4-membered ring with heteroatom',
    13: 'Group IB, IIB (Cu, Zn, Ag, Cd, Au, Hg)',
    14: 'Aromatic ring (A@1)',
    15: 'Halide bonded to heteroatom',
    16: 'Nitrogen with single bond to oxygen (N-O)',
    17: 'Atom is in a 3-membered ring',
    18: 'Group IA (Alkali metals: Li, Na, K, Rb, Cs, Fr)',
    19: 'QHAQH (heteroatom-carbon chain pattern)',
    20: 'Aromatic ring (size >= 5)',
    21: 'Heteroatom in 3-membered ring',
    22: 'NH2 or OH (primary amine or alcohol)',
    23: 'Charged atom (Nitrogen positive or anion)',
    24: 'S (sulfur present)',
    25: 'Amide or thioamide N-C=O/S',
    26: 'Aromatic C-S bond',
    27: 'Nitrogen with 3 bonds, not N+',
    28: '2-ring system fused',
    29: 'Bridged or polycyclic',
    30: 'Atom in ring of size >= 4',
    31: 'F, Cl, Br, I (any halogen)',
    32: 'Si (silicon present)',
    33: 'B (boron present)',
    34: 'Sn or Pb (tin or lead)',
    35: 'D (deuterium / isotope label)',
    36: 'Aryl iodide',
    37: 'Phosphorus-oxygen bond (P-O)',
    38: 'Aryl bromide',
    39: 'Heteroatom (not C, H)',
    40: 'Quaternary carbon (C with 4 non-H)',
    41: 'Aromatic C-Cl',
    42: 'Ester (C(=O)O or OC=O)',
    43: 'Trifluoromethyl (-CF3)',
    44: 'Any ring',
    45: 'Halogen-aliphatic C-Cl',
    46: 'Nitro group (-NO2)',
    47: 'N-oxide (N->O)',
    48: 'Aromatic C-F',
    49: 'Imine or oxime (C=N)',
    50: 'Nitrile (-C#N)',
    51: 'Diazo, azide or N=N',
    52: 'Alkyl chloride (Cl-C)',
    53: 'Isocyanate or isothiocyanate (N=C=O or N=C=S)',
    54: 'Cyclopropane or cyclobutane ring',
    55: 'Alkyl iodide (I-C)',
    56: 'P presence',
    57: 'Sulfonyl (S(=O)=O)',
    58: 'Aromatic',
    59: 'Epoxide',
    60: 'Aryl fluoride',
    61: 'Nitrogen heterocycle (N in ring)',
    62: 'Aldehyde (C=O, H on carbonyl C)',
    63: 'QHAAAQH (pattern)',
    64: 'Alkene (C=C, non-aromatic)',
    65: 'Nitrogen-heteroatom ring',
    66: 'S heterocycle (S in ring)',
    67: 'Phosphate ester (O-P=O)',
    68: 'Se or Te',
    69: 'Alpha-beta unsaturated carbonyl (C=C-C=O)',
    70: 'Halogen on a non-aromatic ring',
    71: 'Hydrazine or hydrazone (N-N)',
    72: 'Azo (Ar-N=N-Ar)',
    73: 'Quaternary nitrogen (N+)',
    74: 'Vinyl halide (X-C=C)',
    75: 'Pyridine ring',
    76: 'Two C=O in molecule',
    77: 'Carbamate (N-C(=O)-O)',
    78: 'Two rings sharing bond (fused)',
    79: 'OH attached to aromatic ring',
    80: 'Guanidine (N-C(=N))',
    81: 'Phosphoric acid or ester',
    82: 'Atom is part of a heteroatom–heteroatom bond',
    83: 'C=O (ketone or aldehyde)',
    84: 'Acrylate / acryloyl (C=CC=O)',
    85: 'Beta-lactam (4-ring with N-C=O)',
    86: 'Sulfone (C-S(=O)(=O)-C)',
    87: 'Amidine (C(=N)N)',
    88: 'Secondary aliphatic amine (NH between 2 C)',
    89: 'Carboxylic acid anhydride (C(=O)OC=O)',
    90: 'Alkyne (C#C)',
    91: 'Atom is in a fused ring system',
    92: 'Oxygen in ring (ether oxygen in ring)',
    93: 'Atom is in a 5-membered heterocycle',
    94: 'Nitrogen in 6-membered ring',
    95: 'Atom is in a 6-membered heterocycle',
    96: 'Oxygen in 5-membered ring',
    97: 'Oxygen in 6-membered ring',
    98: 'Nitrogen in 5-membered ring',
    99: 'N-C-N pattern',
    100: 'Quaternary N',
    101: 'Aromatic C-N bond',
    102: 'HBA (hydrogen bond acceptor)',
    103: 'HBD (hydrogen bond donor)',
    104: 'Phenyl (benzene substituent)',
    105: 'Charged molecule (ionic)',
    106: 'Ketone (C-C(=O)-C)',
    107: 'OH or NH',
    108: 'Oxygen or Sulfur present',
    109: 'Sulfoxide (C-S(=O)-C)',
    110: 'Aromatic nitrogen',
    111: 'Aromatic ring (fused)',
    112: 'Two aromatic rings (biphenyl-like)',
    113: 'Aromatic heterocycle',
    114: 'Aromatic ring with N',
    115: 'Aromatic ring with O or S',
    116: 'Ether (C-O-C)',
    117: 'Tertiary aliphatic amine (N with 3 C)',
    118: 'Secondary amine (NH)',
    119: 'Aliphatic OH (non-phenol)',
    120: 'Sulfide (C-S-C)',
    121: 'OH group',
    122: 'NH group',
    123: 'C=C (alkene)',
    124: 'Amide (N-C=O)',
    125: 'Thiol (SH)',
    126: 'Carbonyl attached to heteroatom',
    127: 'Benzene ring',
    128: 'Cyclopentane ring',
    129: 'Cyclohexane ring',
    130: 'C=O (any carbonyl)',
    131: 'N-H (any)',
    132: 'Nitrogen adjacent to ring',
    133: 'Fluorine present',
    134: 'Nitrogen-containing ring',
    135: 'Double bond (any)',
    136: 'Ring junction',
    137: 'Aromatic atom count >=2',
    138: 'Two heteroatoms',
    139: 'Halogen present',
    140: 'Aromatic C-O bond',
    141: 'Aromatic C=O',
    142: 'Aromatic amine (Ar-NH2 or Ar-NH)',
    143: 'NH2 group',
    144: 'Halogen attached to aromatic ring',
    145: 'Phenol / OH on aromatic ring',
    146: 'C-O (single bond)',
    147: 'Aromatic ring',
    148: 'Methyl group',
    149: 'Nitrogen (any)',
    150: 'Branching (quaternary C or branch point)',
    151: 'Two nitrogen atoms',
    152: 'Two oxygen atoms',
    153: 'Tertiary N (not aromatic)',
    154: 'Ester (C(=O)O)',
    155: 'Carboxylic acid',
    156: 'Sulfonamide or sulfonyl group',
    157: 'Chlorine present',
    158: 'Two rings',
    159: 'Oxygen (any)',
    160: 'Primary amine',
    161: 'N-C=O pattern',
    162: 'Aromatic C-C',
    163: 'Hydroxyl (OH)',
    164: 'Bromine present',
    165: 'Nitrogen present',
    166: 'Aromatic oxygen',
}

# ============================================================
# 2. Helper function to map MACCS bit → description
# ============================================================

def describe_maccs(name):
    idx = int(name.split('_')[-1])
    return MACCS_DESCRIPTIONS.get(idx, f'Bit {idx} (no description)')


# ============================================================
# 3. Apply descriptions to your consensus dataframe
# ============================================================

consensus_df['ChemicalMeaning'] = consensus_df['Feature'].apply(describe_maccs)


# ============================================================
# 4. Display top positive and negative bias features
# ============================================================

print('Top positive bias features (High GU & Low cAMP):')
display(
    consensus_df[consensus_df['ConsensusScore'] > 0]
    [['Feature','ChemicalMeaning','ConsensusScore']]
    .sort_values('ConsensusScore', ascending=False)
    .head(10)
)

print('\nTop negative bias features:')
display(
    consensus_df[consensus_df['ConsensusScore'] < 0]
    [['Feature','ChemicalMeaning','ConsensusScore']]
    .sort_values('ConsensusScore')
    .head(10)
)
